## **MFA Development**

### **Import Complete Subset of Database**

In [91]:
import pandas as pd

# file_path = '/home/shuyan/Research/PPM/GWR_filled_cnn.txt'
file_path = r'd:/Github/Data/data/GWR_filled_cnn_20241027.txt'
df = pd.read_csv(file_path)

# Now you can work with the DataFrame 'df'

C:\Users\xiong\AppData\Local\Temp\ipykernel_20528\872389488.py:5: DtypeWarning:

Columns (17,39,40) have mixed types. Specify dtype option on import or set low_memory=False.



In [ ]:
# # Select categorical columns excluding those with '_status' in their names
# categorical_columns = df.select_dtypes(include=['object']).columns

# filtered_columns = [col for col in categorical_columns if '_status' not in col and col != 'official building number']

# # Create a dictionary to store unique values for each column
# unique_values = {col: df[col].unique() for col in filtered_columns}

# # Find the maximum length of unique values to create a uniform table
# max_len = max(len(values) for values in unique_values.values())

# # Create a DataFrame to store the unique values table
# unique_values_table = pd.DataFrame({col: pd.Series(values) for col, values in unique_values.items()})

# # Display the unique values table
# unique_values_table

In [92]:
# create a dictionary to map old values to new values
energy_source_mapping = {
    'fuel oil': 'oil',
    'natural gas': 'gas',
    'biogas': 'gas',
    'District heating share fossil ≤ 75%': 'district heating',
    'District heating share fossil ≤ 50% (waste heat)': 'district heating',
    'District heating share fossil ≤ 25%': 'district heating',
    'District heating Share fossil > 75%': 'district heating',
    'Electricity (NT)': 'electricity',
    'Electricity (MT)': 'electricity',
    'Electricity (HT)': 'electricity',
    'Electricity (production)': 'electricity',
    'electricity (heat pump)': 'electricity',
    'Thermal solar energy': 'solar energy',
    'wood pellets': 'wood',
    'wood chips': 'wood', 
    'Piece of wood': 'wood',
    'carbon bricks': 'other',
    'gas':'gas',
    'district heating (generic)':'district heating',
    'fuel oil': 'oil',
    'geothermal probe': 'geothermal',
    'Air': 'air',
    'wood (pellets)': 'wood',
    'wood (generic)': 'wood',
    'indefinite': 'other',
    'electricity': 'electricity',
    'earth register': 'geothermal',
    'geothermal (generic)': 'geothermal',
    'Other': 'other',
    'sun (thermal)': 'solar energy',
    'wood (chips)': 'wood',
    'Water (groundwater, surface water, waste water)': 'other',
    'District heating (high temperature)':'district heating',
    'District heating (low temperature)': 'district heating',
    'Waste heat (inside the building)':'other'
}

# List of columns to apply the mapping
columns_to_map = [
    'energy/heat source hot water 1', 'energy/heat source heating 1', 
    'energy/heat source hot water 2', 'energy/heat source heating 2'
]

# Apply the mapping to the specified columns
for column in columns_to_map:
    df[column] = df[column].replace(energy_source_mapping)

# create a dictionary to map old values to new values for heat generator hot water
heat_generator_mapping = {
    'boiler (generic)': 'boiler',
    'Central electric boiler': 'electric boiler',
    'heat pump': 'heat pump',
    'Boiler (generic) for a building': 'boiler',
    'No heat generator': 'no generator',
    'Heat exchangers (including for district heating)': 'heat exchanger',
    'Thermal solar system': 'solar system',
    'Electric storage central heating for a building': 'electric storage heating',
    'Boiler condensing for a building': 'condensing boiler',
    'Electric direct': 'electric direct',
    'Cogeneration plant for a building': 'cogeneration plant',
    'Combined heat and power plant': 'cogeneration plant',
    'Oven': 'oven',
    'Boiler non-condensing': 'non-condensing boiler',
    'Boilers non-condensing for a building': 'non-condensing boiler',
    'Boilers (generic) for multiple buildings': 'boiler',
    'Heat pump for several buildings': 'heat pump',
    'Condensing boilers for several buildings': 'condensing boiler',
    'Cogeneration plant for several buildings': 'cogeneration plant',
    'Thermal solar system for several buildings': 'solar system',
    'Other': 'other'
}

# List of columns to apply the heat generator mapping
heat_generator_columns = [
    'heat generator hot water 1', 'heat generator hot water 2',
    'heat generator heating 1', 'heat generator heating 2'
]

# Apply the heat generator mapping to the specified columns
for column in heat_generator_columns:
    df[column] = df[column].replace(heat_generator_mapping)

In [93]:
# drop columns with _status in their names
df = df.drop(columns=[col for col in df.columns if '_status' in col])

In [94]:
# uncapital all values in the columns for columns with "heat generator" in their names
df.loc[:, df.columns.str.contains('heat generator', case=False)] = df.loc[:, df.columns.str.contains('heat generator', case=False)].apply(lambda x: x.str.lower())
df['heat generator heating 1'].value_counts()

heat generator heating 1
boiler                                                                    1102404
heat pump for a building                                                   341389
no generator                                                               175765
oven                                                                       122358
electric storage heating                                                   118091
heat exchangers (including for district heating) for a building             72167
non-condensing boiler                                                       56335
condensing boiler                                                           41106
electric direct                                                             26419
heat pump                                                                   23179
heat exchangers (including for district heating) for several buildings       8065
solar thermal system for a building                                      

In [95]:
# Define a function to determine building system components based on energy source and heat generator
def get_system_components(energy_source, heat_generator):
    components = []
    
    if isinstance(energy_source, str) and isinstance(heat_generator, str): 
        if energy_source in ['oil', 'fuel oil']:
            components.append('radiators or underfloor heating')
            if 'boiler' in heat_generator:
                components.extend(['oil storage tank', 'oil delivery system', 'flue'])
            if 'condensing' in heat_generator:
                components.append('flue with condensing capability')
        
        if energy_source == 'gas':
            components.extend(['gas supply line', 'radiators or underfloor heating'])
            if 'boiler' in heat_generator or 'CHP' in heat_generator:
                components.append('flue')
            if 'condensing' in heat_generator:
                components.append('flue with condensing capability')
            if 'CHP' in heat_generator:
                components.extend(['CHP unit', 'electrical connection for power generation'])
        
        if energy_source == 'electricity':
            components.append('electrical connection')
            if 'electric boiler' in heat_generator:
                components.append('radiators or underfloor heating')
            if 'storage' in heat_generator:
                components.append('storage heater units')
        
        if energy_source == 'solar energy':
            components.extend(['solar collectors', 'storage tank', 'heat exchanger', 'radiators or underfloor heating'])
            if 'backup' in heat_generator:
                components.append('backup heating system')
        
        if energy_source == 'wood':
            components.extend(['wood storage', 'radiators or underfloor heating'])
            if 'boiler' in heat_generator:
                components.append('flue')
        
        if energy_source == 'geothermal':
            components.extend(['ground loop or boreholes', 'heat pump unit', 'electrical connection', 'radiators or underfloor heating'])
        
        if energy_source == 'district heating':
            components.extend(['connection to district heating network', 'heat exchanger unit', 'radiators or underfloor heating'])
            if 'backup' in heat_generator:
                components.append('backup heating system')
        
        if energy_source == 'other':
            components.append('heat exchanger unit')
        
    return components

# Apply the function to each row in the DataFrame
df['system_components_heating_1'] = df.apply(
    lambda row: get_system_components(row['energy/heat source heating 1'], row['heat generator heating 1']),
    axis=1
)

df['system_components_hot_water_1'] = df.apply(
    lambda row: get_system_components(row['energy/heat source hot water 1'], row['heat generator hot water 1']),
    axis=1
)

# Repeat for the second set of heating and hot water sources if necessary
df['system_components_heating_2'] = df.apply(
    lambda row: get_system_components(row['energy/heat source heating 2'], row['heat generator heating 2']),
    axis=1
)

df['system_components_hot_water_2'] = df.apply(
    lambda row: get_system_components(row['energy/heat source hot water 2'], row['heat generator hot water 2']),
    axis=1
)

In [ ]:
# Define the function to handle the boiler calculation
def has_boiler(row):
    if isinstance(row['heat generator heating 1'], str) and 'boiler' in row['heat generator heating 1']:
        return True
    if isinstance(row['heat generator hot water 1'], str) and 'boiler' in row['heat generator hot water 1']:
        return True
    return False

# define for heat pump
def has_heat_pump(row):
    if isinstance(row['heat generator heating 1'], str) and 'heat pump' in row['heat generator heating 1']:
        return True
    if isinstance(row['heat generator hot water 1'], str) and 'heat pump' in row['heat generator hot water 1']:
        return True
    return False

# To determine the availability of air ducts in buildings based on the provided data, we can use specific criteria related to the building category, building class, and the energy source/heat generator. Here's an enhanced approach:

# Building Category and Class: Certain building categories and classes are more likely to have central HVAC systems with air ducts. For example, non-residential buildings, large residential buildings, commercial buildings, and buildings with more than one apartment.
# Energy Source and Heat Generator: Buildings with specific energy sources and heat generators such as district heating, central electric boilers, heat pumps, and combined heat and power (CHP) systems are more likely to have air ducts.
# Criteria for Air Duct Availability
# We can set up a few rules to determine the likelihood of air ducts being present:

# Building categories like "commercial," "office," "hotel," "school," "university," "hospital," "wholesale and retail buildings," "sports halls," "cultural and leisure purposes," "short-term accommodation," "restaurants and bars," and "residential buildings with multiple apartments" are more likely to have air ducts.
# Building classes such as "Buildings with three or more apartments," "industrial building," "garage building," and "large residential buildings."
# Energy sources like "electricity," "district heating," "geothermal," "solar energy," and "central electric boiler" with corresponding heat generators.
# Modern buildings or those with certifications indicating high energy efficiency.

# Define a function to determine the availability of air ducts
def has_air_ducts(row):
    building_categories_with_ducts = [
        'commercial', 'office building', 'hotel building', 'school', 'university',
        'hospital', 'large residential building', 'wholesale and retail buildings',
        'sports halls', 'cultural and leisure purposes', 'short-term accommodation',
        'restaurants and bars', 'residential buildings with multiple apartments'
    ]
    
    building_classes_with_ducts = [
        'Buildings with three or more apartments', 'industrial building', 
        'garage building', 'large residential buildings'
    ]
    
    energy_sources_with_ducts = ['electricity', 'district heating', 'geothermal', 'solar energy']
    heat_generators_with_ducts = ['central heating', 'heat pump', 'central electric boiler', 'combined heat and power (CHP)']

    if row['building category'] in building_categories_with_ducts:
        return True
    if row['building class'] in building_classes_with_ducts:
        return True
    if row['energy/heat source heating 1'] in energy_sources_with_ducts:
        return True
    if row['heat generator heating 1'] in heat_generators_with_ducts:
        return True
    if row['energy/heat source heating 2'] in energy_sources_with_ducts:
        return True
    if row['heat generator heating 2'] in heat_generators_with_ducts:
        return True
    
    return False

# Apply the function to each row
df['has_air_ducts'] = df.apply(has_air_ducts, axis=1)

In [97]:
# export the results to a new CSV file to local path
df.to_csv(r'D:/Github/Data/data/results_before parametric model_20241209.csv', index=False)

### **Test on Predictive Mathematical Models**

In [1]:
import pandas as pd
# read the results
df = pd.read_csv(r'D:/Github/Data/data/results_before parametric model_20241209.csv')

C:\Users\xiong\AppData\Local\Temp\ipykernel_20308\2413454680.py:3: DtypeWarning: Columns (17,39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:/Github/Data/data/results_before parametric model_20241209.csv')


In [2]:
import pandas as pd
# Define a mapping dictionary for building classes
building_class_mapping = {
    'Building with one apartment': 'residential',
    'Building with two apartments': 'residential',
    'Buildings with three or more apartments': 'residential',
    'Residential buildings for communities': 'residential',

    'office building': 'commercial',
    'wholesale and retail buildings': 'commercial',
    'hotel building': 'commercial',
    'Restaurants and bars in non-residential buildings': 'commercial',

    'School and university buildings, research facilities': 'institutional',
    'Hospitals and specialist healthcare facilities': 'institutional',
    'Churches and other cult buildings': 'institutional',
    'sports halls': 'institutional',
    'museums and libraries': 'institutional',

    # Map remaining buildings to 'other'
    'garage building': 'other',
    'Traffic and communications building without garages': 'other',
    'tanks, silos and storage buildings': 'other',
    'Farm buildings': 'other',
    'Other farm buildings': 'other',
    'Building for animal husbandry': 'other',
    'industrial building': 'other',
    'Other buildings not otherwise specified': 'other',
    'Other buildings for short-term accommodation': 'other',
    'Buildings for cultural and leisure purposes': 'other',
    'Buildings for crop production': 'other',
    'Monuments or listed buildings': 'other',
    'Other buildings for collective housing': 'other'
}

# Define a fallback mapping for building categories
building_category_mapping = {
    'Building with exclusive residential use': 'residential',
    'Non-residential buildings': 'commercial',
    'Other residential buildings (residential buildings with secondary use)': 'residential',
    'special construction': 'other',
    'Building with partial residential use': 'residential',
    'Temporary accommodation': 'other'
}

# Map building class first
df['building class'] = df['building class'].map(building_class_mapping)

# Handle NaN in 'building class' by mapping from 'building category'
df['building class'] = df['building class'].fillna(df['building category'].map(building_category_mapping))

# Replace any remaining NaN values with 'other'
df['building class'] = df['building class'].fillna('other')

In [3]:
from HVAC import calculate_boiler_weight
from HVAC import estimate_boiler_materials
from HVAC import heat_pump_weight
from HVAC import estimate_heat_pump_materials
from HVAC import select_duct_material
from HVAC import calculate_HVAC_pipe_length
from HVAC import calculate_HVAC_pipe_weight
from HVAC import estimate_HVAC_pipe_materials
from HVAC import get_decade
from HVAC import estimate_radiator_by_building_age
from HVAC import estimate_materials_radiator

from plumbing import calculate_water_pipe_length_building
from plumbing import pipe_size_cm
from plumbing import calculate_water_pipe_weight
from plumbing import estimate_water_pipe_materials
from plumbing import select_pipe_material
# from plumbing import decide_pipe_size
from plumbing import calculate_num_bathrooms
from plumbing import estimate_toilet_materials
from plumbing import estimate_sink_materials
from plumbing import estimate_shower_materials
from plumbing import estimate_bathtub_materials
from plumbing import calculate_water_pipe_length_general_buildings

from electrical import calculate_electrical_cable_length_per_apartment
# from electrical import calculate_electrical_cable_length_per_building
from electrical import calculate_electrical_cable_length
from electrical import calculate_electrical_cable_weight
from electrical import estimate_electrical_cable_materials

In [4]:
# Radiator data by decade
radiator_data = {
    '1930s and below': {'type': 'no radiator', 'material': 'cast iron', 'unit_weight': 0, 'lifespan': 20},
    '1940s': {'type': 'cast iron radiator', 'material': 'cast iron', 'unit_weight': 63, 'lifespan': 20},
    '1950s': {'type': 'steel panel radiator', 'material': 'steel', 'unit_weight': 21, 'lifespan': 50},
    '1960s': {'type': 'convector radiator', 'material': 'steel', 'unit_weight': 42, 'lifespan': 30},
    '1970s': {'type': 'column radiator', 'material': 'cast iron', 'unit_weight': 82, 'lifespan': 50},
    '1980s': {'type': 'steel panel radiator', 'material': 'steel', 'unit_weight': 21, 'lifespan': 50},
    '1990s': {'type': 'low surface temperature (LST) radiator', 'material': 'steel', 'unit_weight': 38, 'lifespan': 50},
    '2000s and beyond': {'type': 'low surface temperature (LST) radiator', 'material': 'steel', 'unit_weight': 38, 'lifespan': 50}
}

# Material composition for each radiator type
radiator_material_composition = {
    'cast iron radiator': {'cast iron': 0.95, 'plastic': 0.05},
    'steel panel radiator': {'steel': 0.85, 'plastic': 0.05, 'copper': 0.1},
    'convector radiator': {'steel': 0.8, 'copper': 0.15, 'plastic': 0.05},
    'column radiator': {'cast iron': 0.90, 'plastic': 0.05, 'copper': 0.05},
    'low surface temperature (LST) radiator': {'steel': 0.75, 'copper': 0.15, 'plastic': 0.1}
}

In [5]:
# remove installation year if smaller than construction year
df['2iInst./Renov. heating system'] = df[['2iInst./Renov. heating system','year of construction of the building yyyy']].max(axis=1)
df['2hInst./Renov. hot water supply'] = df[['2hInst./Renov. hot water supply','year of construction of the building yyyy']].max(axis=1)

In [11]:
# exclude demolished buildings
df = df[df['building status'] != 'building demolished']

In [16]:
# Approximate weights for each type of fixture in kilograms
weight_per_toilet = 25      # kg
weight_per_shower = 42      # kg bathome.net/thread-14341-1-1.html
weight_per_sink = 15      # kg
weight_per_bathtub = 100     # kg

# @ air ducts
width = 0.5  # in meters
height = 0.3  # in meters
outer_diameter = 0.5  # in meters (for circular ducts)
wall_thickness = 0.001  # in meters (1 mm)
 
for index, row in df.iterrows():

    # Residential Building
    num_residents = row['9aNumber of residents (EFH/ MFH)'] # need double check
    num_floors = row['number of floors']
    building_height = row['height']
    building_area = row['building area']
    building_age = row['year of construction of the building yyyy']
    # heating system age
    system_age = row['2iInst./Renov. heating system']
    floor_height = building_height/num_floors if num_floors > 0 else 0

    # @ radiator        
    # # Assign 0 radiators for buildings constructed after 2020 and before 1940
    # if system_age > 2020 or system_age < 1940:
    #     df.loc[index, 'radiator weight'] = 0
    #     df.loc[index, 'radiator material weights'] = [0]  # or set to None or an empty list as needed
    #     continue  # Skip to the next row in the DataFrame
    
    if 'radiators or underfloor heating' in row['system_components_heating_1']:
        # assuming 12 kg example weight in kilograms for single radiator
        radiator_unit_weight_kg = radiator_data[get_decade(system_age)]['unit_weight']  # Get unit weight based on building year
        num_radiator = row['number of radiators']
                # Check if number of radiators is zero
        if num_radiator <= 0:
            df.loc[index, 'radiator weight'] = 0
            df.loc[index, 'radiator material weights'] = [0]  # or set to None or an empty list as needed
            continue  # Skip to the next row in the DataFrame
        
        total_radiator_weight = num_radiator * radiator_unit_weight_kg
        
        # Ensure the decade exists in radiator_data
        decade = get_decade(building_age)
        if decade in radiator_data:
            radiator_type, material_weights = estimate_radiator_by_building_age(system_age, radiator_unit_weight_kg)
            df.loc[index, 'radiator weight'] = total_radiator_weight
            df.loc[index, 'radiator material weights'] = [material_weights]
        else:
            # Handle the case where radiator data for the decade is not available
            df.loc[index, 'radiator weight'] = 0
            df.loc[index, 'radiator material weights'] = [0]  

        radiator_type, material_weights = estimate_radiator_by_building_age(system_age, radiator_unit_weight_kg)
        # # radiator_material_weights = estimate_materials_radiator(total_radiator_weight) 
        # df.loc[index, 'radiator weight'] = total_radiator_weight
        # df.loc[index, 'radiator material weights'] = [material_weights]
        # Adjusting assignments in the loop
        df.loc[index, 'radiator weight'] = total_radiator_weight if decade in radiator_data else 0
        df.loc[index, 'radiator material weights'] = [material_weights] if decade in radiator_data else [0]

        
    # @ boiler
    if has_boiler(row):
        # Calculate boiler
        num_boilers=1
        # assumption: https://heatable.co.uk/boiler-advice/what-size-boiler-for-my-home
        # system boiler or conventional boiler
        total_num_rooms = row['number of rooms']
        num_radiator = row['number of radiators']
        boiler_material_weights = estimate_boiler_materials(calculate_boiler_weight(num_radiator))
        df.loc[index,'boiler material weights'] = [boiler_material_weights]
    
    # @ heat pump 
    if has_heat_pump(row):
        # Calculate heat pump
        # https://www.energysavingtrust.org.uk/heat-pumps/heat-pump-sizing
        heat_pump = heat_pump_weight(building_area)
        heat_pump_material_weights = estimate_heat_pump_materials(heat_pump)
        df.loc[index,'heat pump material weights'] = [heat_pump_material_weights]

    # @ air ducts
    # if 'air ducts' in row['has_air_ducts']:
    if row['has_air_ducts']:        

        # Select duct material based on building year
        material, density, material_pct = select_duct_material(building_age)
        total_duct_length_horizontal = row['air duct length']
        HVAC_pipe_total_length = calculate_HVAC_pipe_length(floor_height, total_duct_length_horizontal, building_area, num_floors)
        # HVAC_pipe_weight = calculate_HVAC_pipe_weight(HVAC_pipe_total_length, width, height, outer_diameter, wall_thickness, density, is_rectangular=True)
        HVAC_pipe_weight = calculate_HVAC_pipe_weight(HVAC_pipe_total_length, density)
        HVAC_pipe_material_weights = estimate_HVAC_pipe_materials(HVAC_pipe_weight, material_pct)
        # Calculate weight of a pipe assuming it is rectangular
        # https://www.ductstore.co.uk/acatalog/350x150mm.html#aRDS350_2d150
        # assume steel pipe, kg/m^3 for steel
        df.loc[index, 'air duct total length in m'] = HVAC_pipe_total_length
        df.loc[index,'air duct material weights'] = [HVAC_pipe_material_weights]
   
    # @ plumbing fixtures
    # Calculate plumbing fixtures
    num_bathrooms = calculate_num_bathrooms(total_num_rooms,row['building class'])   
    # Number of each fixture per bathroom
    num_toilets = num_bathrooms
    num_showers = num_bathrooms
    num_sinks = num_bathrooms
    num_bathtubs = num_bathrooms

    toilet_material_weights = estimate_toilet_materials(num_toilets, weight_per_toilet)
    df.loc[index,'toilet material weights'] = [toilet_material_weights]
    
    shower_material_weights = estimate_shower_materials(num_showers, weight_per_shower)
    df.loc[index,'shower material weights'] = [shower_material_weights]
    
    sink_material_weights = estimate_sink_materials(num_sinks, weight_per_sink)
    df.loc[index,'sink material weights'] = [sink_material_weights]
    
    bathtub_material_weights = estimate_bathtub_materials(num_bathtubs, weight_per_bathtub)
    df.loc[index,'bathtub material weights'] = [bathtub_material_weights]

    # @ water pipes
    # Select pipe material based on construction year
    material, density = select_pipe_material(building_age)
    # Assume typical dimensions for the selected material
    if material == "copper":
        diameter = 2.54  # in cm
        wall_thickness = 0.165  # in cm
    elif material == "pex":
        diameter = 1.6  # in cm
        wall_thickness = 0.2  # in cm
    elif material == "galvanized steel":
        diameter = 2.54  # in cm
        wall_thickness = 0.15  # in cm
    
    if row['building class'] == 'residential':
        length_water_pipe_horizontal = row['water pipe length']
        water_pipe_length = calculate_water_pipe_length_building(length_water_pipe_horizontal, num_floors, floor_height, building_area)
    else :
        water_pipe_length = calculate_water_pipe_length_general_buildings(building_area, supply_points=num_bathrooms, avg_distance_to_supply=10, fitting_factor=0.15)
    
    water_pipe_size = pipe_size_cm(num_residents, num_floors)
    # Calculate pipe weight
    water_pipe_weight = calculate_water_pipe_weight(water_pipe_length, density, min(water_pipe_size,diameter), wall_thickness)
    pipe_type = material
    water_pipe_material_weights = estimate_water_pipe_materials(water_pipe_weight, pipe_type)
    df.loc[index, 'water pipe length in m'] = water_pipe_length
    df.loc[index, 'water pipe material weights'] = [water_pipe_material_weights]

    # @ electrical wires
    total_cable_length_horizontal = row['electrical cable length']
    electrical_wire_length = calculate_electrical_cable_length(floor_height, total_cable_length_horizontal, num_floors)
    electrical_wire_copper_weight = calculate_electrical_cable_weight(electrical_wire_length, density=50/1000)
    electrical_wire_material_weights = estimate_electrical_cable_materials(electrical_wire_copper_weight)
    df.loc[index,'electrical cable material weights'] = [electrical_wire_material_weights]

C:\Users\xiong\AppData\Local\Temp\ipykernel_20308\3341995585.py:146: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '(1.034917121700074e-15+16.90147922520393j)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, 'water pipe length in m'] = water_pipe_length
C:\Users\xiong\AppData\Local\Temp\ipykernel_20308\3341995585.py:97: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '(680.018+4.6j)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, 'air duct total length in m'] = HVAC_pipe_total_length


In [17]:
# Identify columns with complex data types
complex_columns = df.select_dtypes(include=['complex128']).columns

# Convert complex columns to float by taking only the real part
for col in complex_columns:
    df[col] = df[col].apply(lambda x: x.real if x.imag == 0 else x).astype('float64')

c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=True)
C:\Users\xiong\AppData\Local\Temp\ipykernel_20308\3393257440.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].apply(lambda x: x.real if x.imag == 0 else x).astype('float64')
c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=True)
C:\Users\xiong\AppData\Local\Temp\ipykernel_20308\3393257440.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


In [18]:
# export the results to a new CSV file
df.to_csv(r'D:/Github/Data/data/calculated_data_20250106.txt', index=False)

In [ ]:
import pandas as pd 
# read the updated DataFrame from the CSV file
# df = pd.read_csv('/media/shuyan/Local Disk (D:)/Github/Data/calculated_data.txt')
# df = pd.read_csv('D:/Github/Data/calculated_data.txt')
# df = pd.read_csv('D:/Github/Data/data/calculated_data_20241112.txt')

In [19]:
df_chunk = df

# Select all columns that contain the phrase 'material weights' in their names
material_columns = df_chunk.filter(like='material weights').columns.tolist()

# Temporary DataFrame to store all expanded columns
expanded_columns_df = pd.DataFrame(index=df_chunk.index)

for col in material_columns:
    if col in df_chunk.columns:
        # Expand each dictionary or list in the column into separate columns
        expanded_df = df_chunk[col].apply(
            lambda x: pd.Series(x[0]) if isinstance(x, list) and len(x) > 0 else 
            (pd.Series(x) if isinstance(x, dict) else pd.Series())
        )
        
        # Rename columns to include the component's name for clarity
        expanded_df = expanded_df.add_suffix(f'_{col}')
        
        # Concatenate expanded columns back to the main DataFrame
        df_chunk = pd.concat([df_chunk, expanded_df], axis=1)
        
        # Drop the original column after expansion
        df_chunk = df_chunk.drop(columns=[col])

# Fill NaNs with 0 only in the expanded columns
df_chunk[expanded_columns_df.columns] = df_chunk[expanded_columns_df.columns].fillna(0)

df_material_weights = df_chunk
df_material_weights

,federal building identifier,canton abbreviation,building status,e building coordinate,n building coordinate,origin of coordinates,building category,building area,building class,area,perimeter,height,energy/heat source hot water 1,heat generator hot water 1,energy/heat source heating 1,heat generator heating 1,number of floors,official building number,living space,number of rooms,cooking equipment,multi-storey apartment,floor,number of radiators,air duct length,water pipe length,number of toilets,number of sink,number of shower,number of bathtub,electrical cable length,number of apartments,year of construction of the building yyyy,energy/heat source hot water 2,heat generator hot water 2,energy/heat source heating 2,heat generator heating 2,energy reference area,building volume,18Construction building,19Floor plan type,2iInst./Renov. heating system,5Area-related outside air volume flow V/AE,year of demolition of the building,9aNumber of residents (EFH/ MFH),2hInst./Renov. hot water supply,system_components_heating_1,system_components_hot_water_1,system_components_heating_2,system_components_hot_water_2,has_air_ducts,radiator weight,water pipe length in m,air duct total length in m,steel_radiator material weights,copper_radiator material weights,plastic_radiator material weights,cast iron_radiator material weights,0_radiator material weights,total_weight_radiator material weights,steel_boiler material weights,copper_boiler material weights,plastic_boiler material weights,mineral wool_boiler material weights,aluminum_boiler material weights,brass_boiler material weights,other electronics_boiler material weights,porcelain_toilet material weights,plastic_toilet material weights,stainless steel_toilet material weights,acrylic_shower material weights,plastic_shower material weights,stainless steel_shower material weights,porcelain_sink material weights,plastic_sink material weights,stainless steel_sink material weights,acrylic_bathtub material weights,plastic_bathtub material weights,cast iron_bathtub material weights,copper_water pipe material weights,steel_water pipe material weights,zinc coating_water pipe material weights,pex_water pipe material weights,copper_electrical cable material weights,pvc_electrical cable material weights,steel_heat pump material weights,aluminum_heat pump material weights,copper_heat pump material weights,plastic_heat pump material weights,refrigerant_heat pump material weights,plastic_air duct material weights,steel_air duct material weights,aluminum_air duct material weights,polyurethane foam_air duct material weights,fiberglass_air duct material weights,mineral wool_air duct material weights
511,1600117,ZH,building consisting,2679546.192,1235476.266,901.0,Building with exclusive residential use,154.0,residential,146.689355,50.003718,6.93,oil,boiler,oil,boiler,2.0,540,200.0,6.0,1.0,1.0,3100.0,11.0,42.426407,113.137085,3.0,3.0,3.0,3.0,1060.660172,1.0,1977.0,NaN,NaN,NaN,NaN,196.0,NaN,difficult,compact,1996.0,0.7,2018.0,0.0,1977.0,"['radiators or underfloor heating', 'oil stora...","['radiators or underfloor heating', 'oil stora...",[],[],False,418.0,174.588897,NaN,28.50,5.7,3.80,NaN,NaN,NaN,42.3375,5.645,2.8225,2.8225,1.129,1.129,0.5645,63.75,7.5,3.75,113.4,6.3,6.3,31.5,9.0,4.5,240.0,30.0,30.0,192.584824+ 0.000000j,NaN+ 0.000000j,NaN+0.000000j,NaN+ 0.000000j,27.803209,33.981700,NaN,NaN,NaN,NaN,NaN,NaN+ 0.000000j,NaN+ 0.000000j,NaN+0.0j,NaN+0.0j,NaN+0.0j,NaN+0.0j
512,1600129,ZH,building consisting,2679327.739,1235459.741,901.0,Building with exclusive residential use,125.0,residential,167.644373,60.979525,5.63,oil,boiler,oil,boiler,3.0,558,150.0,5.0,1.0,1.0,3100.0,8.0,30.618622,85.732141,2.0,2.0,2.0,2.0,765.465545,1.0,1967.0,NaN,NaN,NaN,NaN,328.0,NaN,difficult,stretched,1967.0,0.7,2018.0,0.0,1967.0,"['radiators or underfloor heating', 'oil stora...","['radiators or underfloor heating', 'oil stora...",[],[],False,336.0,137.255744,NaN,33.60,6.3,2.10,NaN,NaN,NaN,38.6250,5.150,2.5750,2.5750,1.030,1.030,0.515

In [20]:
# Identify columns with complex data types
complex_columns = df_material_weights.select_dtypes(include=['complex128']).columns

# Convert complex columns to float by taking only the real part
for col in complex_columns:
    df_material_weights[col] = df_material_weights[col].apply(lambda x: x.real if x.imag == 0 else x).astype('float64')

c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=True)
c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=True)
c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=True)
c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=True)
c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\pandas\core\dtypes\astype.py:134: ComplexWarning: Casting complex values to real discards the imaginary part
  return arr.astype(dtype, copy=T

In [21]:
# Save the updated DataFrame to a new CSV file
# df_expanded.to_csv('/media/shuyan/Local Disk (D:)/Github/Data/material_weight_data_0723.txt')
df_material_weights.to_csv('d:/Github/Data/data/material_weight_data_20250106.txt')

## **MFA Development**

In [22]:
df_BS = pd.read_csv('building_category_building_system_matrix.csv')
df_BS = df_BS.dropna(axis=1, how='all')
# restructure the imported dataframe

# Melt the dataframe
melted_df = pd.melt(df_BS, id_vars=['building system','sub system' ,'component'], var_name='building category', value_name='has_component')

# Filter the rows where the component is present
filtered_df = melted_df[melted_df['has_component'] == True].drop('has_component', axis=1)

# Print the result
filtered_df= filtered_df[['building category', 'building system', 'sub system','component']]
filtered_df

# Load the two dataframes
df1 = filtered_df

# Get the unique component names
component_names = df1['component'].unique()

# column_dict = {}
# for component in component_names:
#     # if not component.startswith('insulation'):
#         columns = [col for col in df2.columns if component in col]
#         column_dict[component] = columns
component_names
# Initialize a dictionary to hold materials for each component
component_materials = {}

# Iterate through columns to find material weights columns
for col in df.columns:
    if 'material weights' in col:
        # Extract the component name from the column name
        component = None
        for name in component_names:
            if col.startswith(name):
                component = name
                break
        if component is not None:
            # Initialize a set to hold unique materials for this component
            materials = set()
            for value in df[col]:
                if isinstance(value, list):  # Ensure value is a list
                    for item in value:
                        if isinstance(item, dict):  # Ensure item is a dictionary
                            materials.update(item.keys())
            # Add an entry to the dictionary for this component
            component_materials[component] = list(materials)

# Print the result
print(component_materials)

{'radiator': ['total_weight', 'steel', 'plastic', 'copper', 'cast iron'], 'boiler': ['steel', 'aluminum', 'plastic', 'other electronics', 'mineral wool', 'copper', 'brass'], 'toilet': ['porcelain', 'plastic', 'stainless steel'], 'shower': ['plastic', 'stainless steel', 'acrylic'], 'sink': ['porcelain', 'plastic', 'stainless steel'], 'bathtub': ['plastic', 'cast iron', 'acrylic'], 'water pipe': ['pex', 'zinc coating', 'copper', 'steel'], 'electrical cable': ['pvc', 'copper'], 'heat pump': ['steel', 'aluminum', 'plastic', 'refrigerant', 'copper'], 'air duct': ['steel', 'aluminum', 'polyurethane foam', 'plastic', 'mineral wool', 'fiberglass']}


In [ ]:
# # save component materials to a dictionary
# component_materials = {'radiator': ['cast iron', 'copper', 'steel', 'plastic'], 'boiler': ['mineral wool', 'brass', 'copper', 'steel', 'aluminum', 'plastic', 'other electronics'], 'water pipe': ['copper', 'zinc coating', 'pex', 'steel'], 'electrical cable': ['pvc', 'copper'], 'toilet': ['porcelain', 'stainless steel', 'plastic'], 'shower': ['plastic', 'glass', 'stainless steel'], 'sink': ['porcelain', 'stainless steel', 'plastic'], 'bathtub': ['plastic', 'acrylic', 'cast iron'], 'air duct': ['fiberglass', 'mineral wool', 'steel', 'aluminum', 'plastic', 'polyurethane foam'], 'heat pump': ['copper', 'steel', 'aluminum', 'refrigerant', 'plastic']}
# # union of all materials from component_materials values
# all_materials = set()
# for materials in component_materials.values():
#     all_materials.update(materials)
# # transform the type of all_materials from dict to list
# all_materials = list(all_materials)
# all_materials

In [23]:
# df3 = df1
# column_dict = component_materials

# for index, row in df3.iterrows():
#     # Iterate over the keys of the dictionary
#     for component in column_dict.keys():
#         # Check if the component exists in the dataframe
#         if component == row['component']:
#             for value in column_dict[row['component']]:
#                 new_row = pd.DataFrame({ 'building category': [row['building category']], 'building system': [row['building system']], 'sub system': [row['sub system']],'component': [row['component']], 'item': value})
#                 df3 = df3.append(new_row, ignore_index=True)

# df3=df3.sort_values(by=['building category','building system','sub system','component']).dropna()
# # add material weights to the new rows
# df3['item']=df3['item']+ '_'+df3['component']+' material weights'

# # df3[df3['building category'] == 'detached house']
# # convert index to lower case
# df3['item'] = df3['item'].str.lower()
# df3





# Assuming df1 is already defined
df3 = df1.copy()
# Assuming component_materials is already defined
column_dict = component_materials

# Iterate through each row in df3
for index, row in df3.iterrows():
    # Iterate over the keys of the dictionary
    for component in column_dict.keys():
        # Check if the component exists in the dataframe
        if component == row['component']:
            for value in column_dict[row['component']]:
                new_row = pd.DataFrame({
                    'building category': [row['building category']],
                    'building system': [row['building system']],
                    'sub system': [row['sub system']],
                    'component': [row['component']],
                    'item': [value]
                })
                df3 = pd.concat([df3, new_row], ignore_index=True)

# Sort the dataframe
df3 = df3.sort_values(by=['building category', 'building system', 'sub system', 'component']).dropna()

# Add material weights to the new rows
df3['item'] = df3['item'] + '_' + df3['component'] + ' material weights'

# Convert index to lower case
df3['item'] = df3['item'].str.lower()

# Display the resulting dataframe
df3


,building category,building system,sub system,component,item
196,commercial,air conditioning system,air conditioning,air duct,steel_air duct material weights
197,commercial,air conditioning system,air conditioning,air duct,aluminum_air duct material weights
198,commercial,air conditioning system,air conditioning,air duct,polyurethane foam_air duct material weights
199,commercial,air conditioning system,air conditioning,air duct,plastic_air duct material weights
200,commercial,air conditioning system,air conditioning,air duct,mineral wool_air duct material weights
...,...,...,...,...,...
296,residential,plumbing system,water heaters,boiler,brass_boiler material weights
282,residential,plumbing system,water supply pipes and fixtures,water pipe,pex_water pipe material weights
283,residential,plumbing system,water supply pipes and fixtures,water pipe,zinc coating_water pipe material weights
284,residential,plumbing system,water supply pipes and fixtures,water pipe,copper_water pipe material weights


In [112]:
# export df3 to csv
df3.to_csv('df3.csv', index=False)

In [14]:
# read df3 to csv
df3 = pd.read_csv('df3.csv')
df3

,building category,building system,sub system,component,item
0,commercial,air conditioning system,air conditioning,air duct,fiberglass_air duct material weights
1,commercial,air conditioning system,air conditioning,air duct,aluminum_air duct material weights
2,commercial,air conditioning system,air conditioning,air duct,steel_air duct material weights
3,commercial,air conditioning system,air conditioning,air duct,mineral wool_air duct material weights
4,commercial,air conditioning system,air conditioning,air duct,polyurethane foam_air duct material weights
...,...,...,...,...,...
149,residential,plumbing system,water heaters,boiler,plastic_boiler material weights
150,residential,plumbing system,water supply pipes and fixtures,water pipe,copper_water pipe material weights
151,residential,plumbing system,water supply pipes and fixtures,water pipe,zinc coating_water pipe material weights
152,residential,plumbing system,water supply pipes and fixtures,water pipe,steel_water pipe material weights


## **Split to Chunk to run**

In [ ]:
df_material_weights =pd.read_csv('d:/Github/Data/data/material_weight_data_20250106.txt')
# Copy the original dataframe
df2 = df_material_weights.copy()

In [10]:
df3 = pd.read_csv('df3.csv')
df3

,building category,building system,sub system,component,item
0,commercial,air conditioning system,air conditioning,air duct,fiberglass_air duct material weights
1,commercial,air conditioning system,air conditioning,air duct,aluminum_air duct material weights
2,commercial,air conditioning system,air conditioning,air duct,steel_air duct material weights
3,commercial,air conditioning system,air conditioning,air duct,mineral wool_air duct material weights
4,commercial,air conditioning system,air conditioning,air duct,polyurethane foam_air duct material weights
...,...,...,...,...,...
149,residential,plumbing system,water heaters,boiler,plastic_boiler material weights
150,residential,plumbing system,water supply pipes and fixtures,water pipe,copper_water pipe material weights
151,residential,plumbing system,water supply pipes and fixtures,water pipe,zinc coating_water pipe material weights
152,residential,plumbing system,water supply pipes and fixtures,water pipe,steel_water pipe material weights


In [27]:
import pandas as pd

# Assuming df3 and df2 are already loaded DataFrames

# Step 1: Remove duplicates from df3 based on 'building category' and 'item'
# to ensure a unique mapping for each category-item pair
unique_df3 = df3.drop_duplicates(subset=['building category', 'item'])

# Step 2: Create a dictionary keyed by a tuple of (building category, item)
item_mapping = (
    unique_df3
    .set_index(['building category', 'item'])[['building system', 'sub system', 'component']]
    .to_dict('index')
)

# Step 3: Define chunk size and prepare an empty list for records
chunk_size = 10000  # Adjust as needed
records = []

# Function to process each chunk of df2
def process_chunk(chunk):
    chunk_records = []
    for _, row in chunk.iterrows():
        year = row['year of construction of the building yyyy']
        category = row['building class']
        
        # Iterate over all columns that might represent items
        for col in row.index:
            # Construct a key based on building class from df2 and the item (col)
            category_item_key = (category, col)
            
            # Check if this category-item pair is in the mapping
            if category_item_key in item_mapping:
                material_weight = row[col].real if isinstance(row[col], complex) else row[col]
                
                # Convert '0j' complex zeros or other invalid values to 0 if needed
                if isinstance(material_weight, (int, float)) and material_weight > 0:
                    system_info = item_mapping[category_item_key]
                    chunk_records.append({
                        'year of construction of the building yyyy': year,
                        'building class': category,
                        'building system': system_info['building system'],
                        'sub system': system_info['sub system'],
                        'component': system_info['component'],
                        'material': col.split('_')[0],
                        'material_weight': material_weight
                    })
    return chunk_records

# Step 4: Process df2 in chunks
for start in range(0, len(df2), chunk_size):
    chunk = df2.iloc[start:start + chunk_size]
    chunk_records = process_chunk(chunk)
    records.extend(chunk_records)

# Step 5: Create the final DataFrame
final_df = pd.DataFrame(records)

# Display the DataFrame
final_df

,year of construction of the building yyyy,building class,building system,sub system,component,material,material_weight
0,1977.0,residential,heating system,heating,radiator,steel,28.500000
1,1977.0,residential,heating system,heating,radiator,copper,5.700000
2,1977.0,residential,heating system,heating,radiator,plastic,3.800000
3,1977.0,residential,heating system,heating,boiler,steel,42.337500
4,1977.0,residential,heating system,heating,boiler,copper,5.645000
...,...,...,...,...,...,...,...
49013162,1998.0,other,electrical system,electrical wiring and cables,electrical cable,copper,376.429500
49013163,1998.0,other,electrical system,electrical wiring and cables,electrical cable,pvc,460.080500
49013164,1997.0,other,plumbing system,drainage and waste pipes,water pipe,copper,26.120784
49013165,1997.0,other,electrical system,electrical wiring and cables,electrical cable,copper,376.429500


In [28]:
# Save the final DataFrame to a CSV file
final_df.to_csv(f'd:/Github/Data/final_material_data_20250106.csv', index=False)

# **Sankey Diagram**

## Separate Sankey Diagram

In [ ]:
# read final_df
import pandas as pd
df_final = pd.read_csv(r'd:/Github/Data/final_material_data_20250106.csv')
df_final

,year of construction of the building yyyy,building class,building system,sub system,component,material,material_weight
0,2021.0,residential,heating system,heating,radiator,steel,28.500
1,2021.0,residential,heating system,heating,radiator,copper,5.700
2,2021.0,residential,heating system,heating,radiator,plastic,3.800
3,2021.0,residential,plumbing system,water heaters,boiler,steel,48.525
4,2021.0,residential,plumbing system,water heaters,boiler,copper,6.470
...,...,...,...,...,...,...,...
67488065,1997.0,other,plumbing system,plumbing fixtures,sink,plastic,519.000
67488066,1997.0,other,plumbing system,plumbing fixtures,sink,stainless steel,259.500
67488067,1997.0,other,plumbing system,plumbing fixtures,bathtub,acrylic,13840.000
67488068,1997.0,other,plumbing system,plumbing fixtures,bathtub,plastic,1730.000


In [29]:
df_final = final_df

In [30]:
# exclude historical buildings before 1930
df_final_wt_hb = df_final[df_final['year of construction of the building yyyy'] >= 1930]
# # exclude buildings newer than 2020
# df_final_wt_hb = df_final_wt_hb[df_final_wt_hb['year of construction of the building yyyy'] <= 2020]
df_final_wt_hb

,year of construction of the building yyyy,building class,building system,sub system,component,material,material_weight
0,1977.0,residential,heating system,heating,radiator,steel,28.500000
1,1977.0,residential,heating system,heating,radiator,copper,5.700000
2,1977.0,residential,heating system,heating,radiator,plastic,3.800000
3,1977.0,residential,heating system,heating,boiler,steel,42.337500
4,1977.0,residential,heating system,heating,boiler,copper,5.645000
...,...,...,...,...,...,...,...
49013162,1998.0,other,electrical system,electrical wiring and cables,electrical cable,copper,376.429500
49013163,1998.0,other,electrical system,electrical wiring and cables,electrical cable,pvc,460.080500
49013164,1997.0,other,plumbing system,drainage and waste pipes,water pipe,copper,26.120784
49013165,1997.0,other,electrical system,electrical wiring and cables,electrical cable,copper,376.429500


In [31]:
# remove bathtubs for all buildings except residential buildings
df_final_wt_hb = df_final_wt_hb[~((df_final_wt_hb['component'] == 'bathtub') & (df_final_wt_hb['building class'] != 'residential'))]
# add stainless steel and steel to be the same material
df_final_wt_hb['material'] = df_final_wt_hb['material'].replace({'stainless steel': 'steel'})

In [32]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Aggregating material weight by building cSlass, material, material weight
material_weight_sum = df_final_wt_hb.groupby(['building class', 'material'])['material_weight'].sum().reset_index()
# export the final dataframe to csv in a typicaL path
material_weight_sum.to_csv('material_weight_sum_20250106.csv', index=False)

In [33]:
import plotly.express as px
import pandas as pd
material_weight_sum = df_final_wt_hb.groupby(['building class'])['material_weight'].sum().reset_index()

# Building numbers summary
building_numbers = {
    'residential': 1844452,
    'other': 1128274,
    'commercial': 167447,
    'institutional': 31681
}

# Map building class to building numbers
material_weight_sum['building_numbers'] = material_weight_sum['building class'].map(building_numbers)

# Plotting
fig = px.scatter(
    material_weight_sum,
    x='building_numbers',  # Switched x-axis
    y='material_weight',   # Switched y-axis
    color='building class',
    size=[10] * len(material_weight_sum),  # Uniform point sizes
    text='building class',  # Add labels for building classes
    labels={
        'building_numbers': 'Building Numbers',
        'material_weight': 'Total Material Weight',
        'building class': 'Building Class'
    },
    title='Scatter Plot of Building Numbers vs Total Material Weight',
    color_discrete_map={
        'commercial': '#1f77b4',  # Updated to light blue
        'residential': '#2ca02c',  # Updated to a brighter green
        'other': '#ff7f0e',  # Updated to orange
        'institutional': '#d62728'  # Updated to bright red
    }
)

# Update layout to remove grid and customize appearance
fig.update_traces(textposition='top center', marker=dict(line=dict(width=1, color='white')))
fig.update_layout(
    template='plotly_white',
    legend_title=dict(text='Building Class'),
    xaxis=dict(showgrid=False),  # Remove x-axis grid
    yaxis=dict(showgrid=False),  # Remove y-axis grid
    plot_bgcolor='white'  # Set background color to white
)

# Export to SVG
fig.write_image('scatter_plot_building_numbers_material_weight_no_grid.svg')

# Show plot
fig.show()

In [35]:
# read the final dataframe from csv
material_weight_sum = pd.read_csv('material_weight_sum_20250106.csv')

building_numbers = {
    'residential': 1844452,
    'other': 1128274,
    'commercial': 167447,
    'institutional': 31681
}

# Create DataFrame
df = material_weight_sum
# rename building class column
df = df.rename(columns={'building class': 'building_class'})
# Map building numbers to the DataFrame
df['building_numbers'] = df['building_class'].map(building_numbers)

import plotly.express as px

# Plotting with Plotly
fig = px.scatter(
    df,
    x='building_numbers',
    y='material_weight',
    color='material',
    symbol='building_class',
    # size='material_weight',
    labels={
        'building_numbers': 'Building Numbers',
        'material_weight': 'Material Weight',
        'material': 'Material',
        'building_class': 'Building Class'
    },
    title='Scatter Plot of Material Weight vs Building Numbers'
)

# template with white background and no gridlines
fig.update_layout(template='plotly_white', xaxis_showgrid=False, yaxis_showgrid=False)
fig.show()

In [36]:
# aggregate the material weights by building class
df_final_agg = df_final_wt_hb.groupby(['building class', 'material'], as_index=False)['material_weight'].sum()
# exclude building class 'other'
df_final_agg = df_final_agg[df_final_agg['building class'] != 'other']

In [37]:
import plotly.express as px

# Create the plotly figure
fig = px.bar(
    df_final_agg,
    x='material',
    y='material_weight',
    color='building class',
    barmode='group',
    title='Material Weights by Building Class',
    color_discrete_sequence=px.colors.qualitative.Pastel,  # Pastel color sequence
)

# Update layout for appearance
fig.update_layout(
    template='plotly_white',  # white background grid similar to seaborn "whitegrid"
    xaxis_title="Material",
    yaxis_title="Material Weight",
    xaxis=dict(tickangle=45),
    width=1200,
    height=600
)

# Display the figure
fig.show()

# Export as SVG
fig.write_image("material_weights.svg")


In [38]:
import plotly.express as px

# Create the plotly figure
fig = px.bar(
    df_final_agg,
    x='material',
    y='material_weight',
    color='building class',
    barmode='group',
    title='Material Weights by Building Class',
    color_discrete_sequence=px.colors.qualitative.Pastel,  # Pastel color sequence
)

# Update layout for appearance
fig.update_layout(
    template='plotly_white',
    xaxis_title="Material",
    yaxis_title="Material Weight",
    xaxis=dict(tickangle=45),
    width=1200,
    height=600
)

# Change y-axis to log scale
fig.update_yaxes(type='log')

# Display the figure
fig.show()

# Export as SVG
fig.write_image("material_weights_log_scale.svg")

In [39]:
# Define component lifespan
component_lifespan = {
    'electrical cable': 50,
    'boiler': 35,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 35,
    'heat pump': 30,
    'toilet': 50,
    'shower': 30,
    'sink': 50,
    'bathtub': 50
}

# Add end-of-life year for each component
df_final_wt_hb['end_of_life_year'] = df_final_wt_hb['year of construction of the building yyyy'] + df_final_wt_hb['component'].map(component_lifespan)

# Group by decade of end-of-life year and material type, summing material weights
df_final_wt_hb['end_of_life_decade'] = (df_final_wt_hb['end_of_life_year'] // 10) * 10
material_availability = df_final_wt_hb.groupby(['end_of_life_decade', 'material'])['material_weight'].sum().unstack(fill_value=0)

In [40]:
material_availability

material,acrylic,aluminum,brass,cast iron,copper,fiberglass,mineral wool,other electronics,pex,plastic,polyurethane foam,porcelain,pvc,refrigerant,steel,zinc coating
end_of_life_decade,,,,,,,,,,,,,,,,
1950.0,0.0,0.000000e+00,0.000,0.00,0.000000e+00,0.000000,5.469570e+05,0.0000,0.000000e+00,0.000000e+00,0.000000e+00,0.00,0.000000e+00,0.000000,1.276233e+06,0.000000
1960.0,31366175.4,4.085164e+05,123835.164,7204.50,1.537326e+06,0.000000,1.852135e+06,61917.5820,0.000000e+00,2.521828e+06,0.000000e+00,0.00,0.000000e+00,71170.299385,1.293934e+07,0.000000
1970.0,40829934.6,1.037295e+06,308548.533,16280.10,3.558007e+06,0.000000,2.215823e+06,154274.2665,0.000000e+00,4.133392e+06,7.681661e+05,0.00,0.000000e+00,124823.483514,2.441465e+07,0.000000
1980.0,99124174.6,1.060296e+06,330817.926,8327506.25,2.243139e+07,0.000000,8.270448e+05,165408.9630,0.000000e+00,1.749570e+07,1.621609e+06,32358901.50,2.343435e+07,61275.198051,5.196839e+07,286164.916718
1990.0,130357819.4,8.962819e+05,225235.587,10846957.00,3.184726e+07,0.000000,5.630890e+05,112617.7935,0.000000e+00,2.230695e+07,1.646652e+06,46498573.50,3.656550e+07,44797.278209,5.344282e+07,358969.654319
2000.0,103372170.6,8.188376e+05,220600.289,8703267.00,5.042068e+07,72304.988863,5.515007e+05,110300.1445,0.000000e+00,2.008193e+07,5.693482e+05,48623315.25,5.893739e+07,88966.815944,5.205227e+07,345852.000248
2010.0,118516530.8,6.419036e+05,205622.035,11655471.40,8.335573e+07,110674.100157,5.140551e+05,102811.0175,0.000000e+00,2.610232e+07,0.000000e+00,69324982.00,9.934156e+07,81401.868699,6.616838e+07,481506.410357
2020.0,90406618.0,4.716920e+05,156409.005,9015770.00,1.313163e+08,51710.201027,3.910225e+05,78204.5025,0.000000e+00,2.063796e+07,0.000000e+00,54064185.75,1.013172e+08,65893.210214,1.420882e+07,0.000000
2030.0,86952366.6,1.195565e+06,151297.253,6748860.00,1.115447e+08,0.000000,3.782431e+05,75648.6265,0.000000e+00,1.793579e+07,0.000000e+00,40474582.50,8.896773e+07,261066.937720,1.541004e+07,0.000000


In [42]:
# only take material end of decade from 2020 onwards
material_availability_future = material_availability.loc[2030:]

In [43]:
# # Create a vertical bar chart
# material_availability.plot(kind='bar', stacked=True, figsize=(10, 5))
# plt.title("Circular Streams of Available Materials in the Upcoming Decades")
# plt.xlabel("End-of-Life Decade")
# plt.ylabel("Material Weight (in units)")
# # remove grid
# plt.grid(False)
# plt.legend(title="Material Type", bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Reset index to get 'end_of_life_decade' as a column
material_availability_reset = material_availability_future.reset_index()

# Melt the DataFrame to long format
material_availability_melted = material_availability_reset.melt(
    id_vars='end_of_life_decade',
    var_name='Material',
    value_name='Material Weight'
)
import plotly.express as px

# Create the stacked bar chart
fig = px.bar(
    material_availability_melted,
    x='end_of_life_decade',
    y='Material Weight',
    color='Material',
    title="Circular Streams of Available Materials in the Upcoming Decades",
    labels={'end_of_life_decade': 'End-of-Life Decade', 'Material Weight': 'Material Weight (in units)'},
    width=800,
    height=500,
    color_discrete_sequence=px.colors.qualitative.Pastel  # Use Pastel palette
)
fig.update_layout(barmode='stack')

# Cleaner layout
fig.update_layout(
    template='plotly_white',
    xaxis_title='End-of-Life Decade',
    yaxis_title='Material Weight (in units)',
    legend_title='Material Type',
    showlegend=True
)

fig.show()

# Export the figure as SVG
fig.write_image('material_availability.svg')


### show only the steel as example

In [44]:
# Filter the DataFrame to only include steel material
steel_df = df_final_wt_hb[df_final_wt_hb['material'] == 'steel']

steel_df['construction_decade'] = (steel_df['year of construction of the building yyyy'] // 10) * 10
# plot bar chart of steel material weights by construction decade using plotly
import plotly.express as px
# Group the steel data by construction decade and material type, summing the material weights
steel_stock_by_decade = steel_df.groupby(['construction_decade'])['material_weight'].sum().reset_index()

# Create the bar chart using Plotly
fig = px.bar(
    steel_stock_by_decade,
    x='construction_decade',
    y='material_weight',
    title='Availability of Steel Material Stock by Construction Year',
    labels={'construction_decade': 'Construction Decade', 'material_weight': 'Material Weight'},
    width=800,
    height=500,
    color_discrete_sequence=px.colors.qualitative.Pastel  # Use Pastel palette
)

# I want a cleaner layout
fig.update_layout(
    template='plotly_white',
    xaxis_title='Construction Decade',
    yaxis_title='Material Weight',
    showlegend=False
)
fig.show()
fig.write_image('steel_material_availability.svg')


C:\Users\xiong\AppData\Local\Temp\ipykernel_20308\2031551459.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [45]:
# exclude end of life year before 2020
steel_df = steel_df[steel_df['end_of_life_year'] > 2020]
# Group the steel data by end of life decade and construction decade, summing the material weights
steel_transfer = steel_df.groupby(['end_of_life_decade', 'construction_decade'])['material_weight'].sum().unstack(fill_value=0)
# # Create the stacked bar chart
# steel_transfer.plot(kind='bar', stacked=True, figsize=(10, 6))
# plt.xlabel('End of Life Decade')
# plt.ylabel('Material Weight (units)')
# plt.title('Steel Material Transfer by Decade')
# plt.grid(False)
# plt.legend(title='Decade of Construction')
import plotly.express as px

# Reset index to get 'end_of_life_decade' as a column
steel_transfer_reset = steel_transfer.reset_index()

# Melt the DataFrame to long format
steel_transfer_melted = steel_transfer_reset.melt(
    id_vars='end_of_life_decade',
    var_name='construction_decade',
    value_name='material_weight'
)

# Create the stacked bar chart using Plotly
fig = px.bar(
    steel_transfer_melted,
    x='end_of_life_decade',
    y='material_weight',
    color='construction_decade',
    title='Steel Material Transfer by Decade',
    color_discrete_sequence=px.colors.qualitative.Pastel,  # Use Pastel palette
    labels={
        'end_of_life_decade': 'End of Life Decade',
        'material_weight': 'Material Weight (units)',
        'construction_decade': 'Construction Decade'
    },
    barmode='stack'
)
# I want a cleaner layout
fig.update_layout(
    template='plotly_white',
    xaxis_title='End of Life Decade',
    yaxis_title='Material Weight (units)',
    legend_title='Construction Decade',
    showlegend=True
)
fig.show()
# export the figure to svg
fig.write_image('steel_transfer.svg')

### sankey plot

In [41]:
import pandas as pd
import plotly.graph_objects as go

def genSankey(df, cat_cols, value_col, title):
    # Create labels
    labels = []
    for cat in cat_cols:
        labels.extend(list(df[cat].unique()))
    labels = list(dict.fromkeys(labels))  # Remove duplicates while maintaining order

    # Create a dictionary to map labels to indices
    label_dict = {label: i for i, label in enumerate(labels)}

    # Create source and target indices
    sources = []
    targets = []
    values = []

    for i in range(len(cat_cols) - 1):
        cat1 = cat_cols[i]
        cat2 = cat_cols[i + 1]
        for j, row in df.iterrows():
            sources.append(label_dict[row[cat1]])
            targets.append(label_dict[row[cat2]])
            values.append(row[value_col])

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
        )
    )])

    fig.update_layout(title_text=title, font_size=10)
    return fig

# remove building class 'other'
df_final_wt_hb = df_final_wt_hb[df_final_wt_hb['building class'] != 'other']
# aggregate the material weights
df_agg = df_final_wt_hb.groupby(['building class', 'building system', 'component','material'],as_index=False).agg(material_weight=('material_weight','sum'))


fig = genSankey(df_agg, cat_cols=['building class', 'building system', 'component', 'material'], value_col='material_weight', title='Demo Material Flows Sankey')
config = {
    'toImageButtonOptions': {
        'format': 'svg',  # one of png, svg, jpeg, webp
        'filename': 'material_flows_sankey',
        'height': 1080,
        'width': 1920,
        'scale': 2  # Multiply title/legend/axis/canvas sizes by this factor
    }
}
fig.show(config=config)
# export offline plot
# fig.write_html('D:/Github/Data/material_flows_sankey.html', auto_open=True)
# export the figure to svg
fig.write_image('material_flows_sankey.svg')


### additional plots

In [ ]:
import pandas as pd

df = df_final
# Define component lifespan
component_lifespan = {
    'electrical cable': 30,
    'boiler': 20,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 40,
    'heat pump': 15,
    'toilet': 30,
    'shower': 20,
    'sink': 30,
    'bathtub': 30
}

# Define future decades
future_decades = [f"{2024 + i*10}-{2034 + i*10}" for i in range(5)]  # You can adjust the range as needed

# Calculate end-of-life decade
def calculate_end_of_life(row):
    lifespan = component_lifespan.get(row['component'], 0)
    return row['year of construction of the building yyyy'] + lifespan

df['end_of_life_year'] = df.apply(calculate_end_of_life, axis=1)

# Create columns for each future decade and initialize with zero
for decade in future_decades:
    df[decade] = 0

# Populate the future decade columns with material weights based on end-of-life year
for index, row in df.iterrows():
    start_year= row['year of construction of the building yyyy']
    end_year = row['end_of_life_year']
    
    for decade in future_decades:
        start, end = map(int, decade.split('-'))
        
        if start_year < end <= end_year:
            df.at[index, decade] = row['material_weight']


df.drop(columns=['year of construction of the building yyyy'], inplace=True)
# Group by future_decade and other relevant columns and sum the material weights
df_aggregated = df.groupby(['building class', 'building system', 'component', 'material','2024-2034','2034-2044','2044-2054','2054-2064','2064-2074']).agg({'material_weight': 'sum'}).reset_index()
df_aggregated

In [ ]:
import pandas as pd

df = df_final
# Define component lifespan
component_lifespan = {
    'electrical cable': 30,
    'boiler': 20,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 40,
    'heat pump': 15,
    'toilet': 30,
    'shower': 20,
    'sink': 30,
    'bathtub': 30
}

# Define future decades
future_decades = [f"{2024 + i*10}-{2034 + i*10}" for i in range(5)]  # You can adjust the range as needed

# Calculate end-of-life decade
def calculate_end_of_life(row):
    lifespan = component_lifespan.get(row['component'], 0)
    return row['year of construction of the building yyyy'] + lifespan

df['end_of_life_year'] = df.apply(calculate_end_of_life, axis=1)

# Create columns for each future decade and initialize with zero
for decade in future_decades:
    df[decade] = 0

# Populate the future decade columns with material weights based on end-of-life year
for index, row in df.iterrows():
    start_year= row['year of construction of the building yyyy']
    end_year = row['end_of_life_year']
    
    for decade in future_decades:
        start, end = map(int, decade.split('-'))
        
        if start_year < end <= end_year:
            df.at[index, decade] = row['material_weight']


df.drop(columns=['year of construction of the building yyyy'], inplace=True)
# Group by future_decade and other relevant columns and sum the material weights
df_aggregated = df.groupby(['building class', 'building system', 'component', 'material','2024-2034','2034-2044','2044-2054','2054-2064','2064-2074']).agg({'material_weight': 'sum'}).reset_index()
df_aggregated
import matplotlib.pyplot as mat

# Melt the dataframe to
df_melted = df_aggregated.melt(id_vars=['building class', 'building system', 'component', 'material'], 
                               value_vars=future_decades, 
                               var_name='Decade', 
                               value_name='Material Weight')

# Plot the bar chart
plt.figure(figsize=(14, 8))
sns.barplot(x='Decade', y='Material Weight', hue='material', data=df_melted)
plt.title('Future Availability of Materials by Decade')
plt.ylabel('Material Weight')
plt.xlabel('Decade')
plt.xticks(rotation=45)
plt.legend(title='Material',= bbox_to_anchor'Material',= bbox_to_anchor(
plt.tight_layout()
plt.show()
plt.tight_layout()
plt.show()1.05,= 1), loc(1.05,= 1),'upper loc left')='upper left')
df_melted have = df_aggregated.melt(id_vars= a['buildin
plt.ylabel('Material Weight')
plt.xlabel('Decade')
plt.xticks(rotation=45)g c
                               var_name='Decade', 
plt.title('Future Availability of Materials by Decade')

sns.barplot(x='Decade', y='Material Weight', hue='material', data=df_melted)
plt.figure(figsize=(14, 8))
# Plot the bar chart                               value_name='Material Weight')
lass', long 'building system', 'component', f
                               value_vars=future_decades, ormat 'material'],  suitable for bar plot

# Melt the dataframe to have a long format suitable for bar plotplotlib.pyplot plt as plt

SyntaxError: unmatched ')' (3702947371.py, line 67)

In [ ]:
# export the final dataframe to csv in a typicaL path
df_aggregated.to_csv('/media/shuyan/Local Disk (D:)/Github/Data/final_data_aggregated.csv', index=False)

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Melt the dataframe to long format
df_melted = df_aggregated.melt(id_vars=['building class', 'building system', 'component', 'material'], 
                               var_name='decade', value_name='material_weight')

# Shift the decades to represent transitions
df_melted['next_decade'] = df_melted.groupby(['building class', 'building system', 'component', 'material'])['decade'].shift(-1)
df_melted.dropna(inplace=True)  # Drop rows where next_decade is NaN
df_melted['next_decade'] = df_melted['next_decade'].astype(str)

# Combine material and decades for unique labels in Sankey
df_melted['source'] = df_melted['material'] + '_' + df_melted['decade']
df_melted['target'] = df_melted['material'] + '_' + df_melted['next_decade']

# Remove self-references
df_melted = df_melted[df_melted['source'] != df_melted['target']]

# Define the Sankey generation function
def genSankey(df, source_col, target_col, value_col, title):
    # Create labels
    labels = list(set(df[source_col].unique()).union(set(df[target_col].unique())))
    
    # Create a dictionary to map labels to indices
    label_dict = {label: i for i, label in enumerate(labels)}

    # Map source and target labels to indices
    df['source_id'] = df[source_col].apply(lambda x: label_dict[x])
    df['target_id'] = df[target_col].apply(lambda x: label_dict[x])

    # Create the Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels,
        ),
        link=dict(
            source=df['source_id'],
            target=df['target_id'],
            value=df[value_col],
        )
    )])

    fig.update_layout(title_text=title, font_size=10)
    return fig

# Generate the Sankey diagram
fig = genSankey(df_melted, source_col='source', target_col='target', value_col='material_weight', title='Future Availability of Material Flows by Decade')

# Configurations for exporting
config = {
    'toImageButtonOptions': {
        'format': 'svg',  # one of png, svg, jpeg, webp
        'filename': 'material_flows_by_decade',
        'height': 1080,
        'width': 1920,
        'scale': 2  # Multiply title/legend/axis/canvas sizes by this factor
    }
}

# Show the figure with the given configuration
fig.show(config=config)

## Combined Sankey Diagram

In [ ]:
import pandas as pd
import plotly.graph_objects as go
df = df_final

# Define component lifespan
component_lifespan = {
    'electrical cable': 30,
    'boiler': 20,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 40,
    'heat pump': 15,
}

# Define future decades
future_decades = [f"{2024 + i*10}-{2034 + i*10}" for i in range(5)]  # You can adjust the range as needed
# Calculate end-of-life decade
def calculate_end_of_life(row):
    lifespan = component_lifespan.get(row['component'], 0)
    return row['year of construction of the building yyyy'] + lifespan

df['end_of_life_year'] = df.apply(calculate_end_of_life, axis=1)
# Define a function to create decade columns
def assign_decade(year):
    if year >= 2024 and year < 2034:
        return '2024-2034'
    elif year >= 2034 and year < 2044:
        return '2034-2044'
    elif year >= 2044 and year < 2054:
        return '2044-2054'
    elif year >= 2054 and year < 2064:
        return '2054-2064'
    elif year >= 2064 and year < 2074:
        return '2064-2074'
    else:
        return 'before 2024'

# Apply function to create the decade column
df['end_of_life_decade'] = df['end_of_life_year'].apply(assign_decade)
# Group by future_decade and other relevant columns and sum the material weights
df_aggregated = df.groupby(['building class', 'building system', 'component', 'material','end_of_life_decade']).agg({'material_weight': 'sum'}).reset_index()

df= df_aggregated
# Prepare the data for the Sankey diagram
# We need to create lists of unique labels for nodes and a way to map them to the source and target of links

# Create unique lists of labels
building_category = df['building class'].unique().tolist()
building_system = df['building system'].unique().tolist()
component = df['component'].unique().tolist()
material = df['material'].unique().tolist()
end_of_life_decade = df['end_of_life_decade'].unique().tolist()

labels = building_category + building_system + component + material + end_of_life_decade

# Create mappings from labels to indices
label_to_index = {label: index for index, label in enumerate(labels)}

# Create source and target lists for the Sankey diagram
source = []
target = []
value = []

# Mapping building category to building system
for _, row in df.iterrows():
    source.append(label_to_index[row['building class']])
    target.append(label_to_index[row['building system']])
    value.append(row['material_weight'])

# Mapping building system to component
for _, row in df.iterrows():
    source.append(label_to_index[row['building system']])
    target.append(label_to_index[row['component']])
    value.append(row['material_weight'])

# Mapping component to material
for _, row in df.iterrows():
    source.append(label_to_index[row['component']])
    target.append(label_to_index[row['material']])
    value.append(row['material_weight'])

# Mapping material to end of life decade
for _, row in df.iterrows():
    source.append(label_to_index[row['material']])
    target.append(label_to_index[row['end_of_life_decade']])
    value.append(row['material_weight'])

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Building Material Flow Sankey Diagram", font_size=10)
fig.show()


# **Add Reuse/Recycle/Refurbish/Disposal post flows**

In [ ]:
import pandas as pd

# Sample data
data = {
    "year_of_construction": [1530.0, 1530.0, 1530.0, 1530.0, 1530.0, 2021.0, 2021.0, 2021.0, 2021.0, 2021.0],
    "building_class": ["housing"]*10,
    "building_system": ["electrical system", "electrical system", "heating system", "heating system", "heating system", "plumbing system", "plumbing system", "plumbing system", "plumbing system", "plumbing system"],
    "sub_system": ["electrical wiring and cables", "electrical wiring and cables", "heating", "heating", "heating", "drainage and waste pipes", "water supply pipes and fixtures", "water supply pipes and fixtures", "water supply pipes and fixtures", "water supply pipes and fixtures"],
    "component": ["electrical cable", "electrical cable", "boiler", "boiler", "boiler", "water pipe", "water pipe", "water pipe", "water pipe", "water pipe"],
    "material": ["copper", "pvc", "aluminum", "brass", "copper", "zinc coating", "copper", "pex", "steel", "zinc coating"],
    "material_weight": [0.002017, 0.078648, 1.030000, 1.030000, 5.150000, 0.000000, 0.000000, 16.050662, 0.000000, 0.000000]
}

df = pd.DataFrame(data)

# Define typical lifespans for components
component_lifespan = {
    'electrical cable': 30,
    'boiler': 20,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 40,
    'heat pump': 15,
}

# Define percentage distributions based on component
component_distributions = {
    "electrical cable": {"reuse": 0.10, "recycle": 0.40, "refurbish": 0.10, "disposal": 0.40},
    "boiler": {"reuse": 0.05, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.35},
    "water pipe": {"reuse": 0.20, "recycle": 0.30, "refurbish": 0.10, "disposal": 0.40},
    "air duct": {"reuse": 0.15, "recycle": 0.45, "refurbish": 0.15, "disposal": 0.25},
    "radiator": {"reuse": 0.05, "recycle": 0.25, "refurbish": 0.05, "disposal": 0.65},
    "heat pump": {"reuse": 0.05, "recycle": 0.55, "refurbish": 0.15, "disposal": 0.25},
}

# Define percentage distributions based on material
material_distributions = {
    "copper": {"reuse": 0.20, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.20},
    "pvc": {"reuse": 0.10, "recycle": 0.40, "refurbish": 0.10, "disposal": 0.40},
    "aluminum": {"reuse": 0.15, "recycle": 0.60, "refurbish": 0.10, "disposal": 0.15},
    "brass": {"reuse": 0.25, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.15},
    "zinc coating": {"reuse": 0.10, "recycle": 0.30, "refurbish": 0.10, "disposal": 0.50},
    "pex": {"reuse": 0.10, "recycle": 0.20, "refurbish": 0.10, "disposal": 0.60},
    "steel": {"reuse": 0.15, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.25},
}

# Current year
current_year = 2024

# Calculate age and determine post-use classification based on component and material
def classify_component(row):
    component = row['component']
    material = row['material']
    age = current_year - row['year_of_construction']
    lifespan = component_lifespan.get(component, 0)
    component_dist = component_distributions.get(component, {"reuse": 0, "recycle": 0, "refurbish": 0, "disposal": 0})
    material_dist = material_distributions.get(material, {"reuse": 0, "recycle": 0, "refurbish": 0, "disposal": 0})
    
    if age < lifespan:
        # Multiply component and material distributions
        reuse = component_dist['reuse'] * material_dist['reuse']
        recycle = component_dist['recycle'] * material_dist['recycle']
        refurbish = component_dist['refurbish'] * material_dist['refurbish']
        disposal = component_dist['disposal'] * material_dist['disposal']
        
        # Normalize to ensure the sum is 1
        total = reuse + recycle + refurbish + disposal
        row['reuse'] = reuse / total
        row['recycle'] = recycle / total
        row['refurbish'] = refurbish / total
        row['disposal'] = disposal / total
    else:
        row['reuse'] = 0
        row['recycle'] = 0
        row['refurbish'] = 0
        row['disposal'] = 1
    
    return row

df = df.apply(classify_component, axis=1)

df


In [ ]:
import pandas as pd
import plotly.graph_objects as go


# df = pd.DataFrame(data)
df = df_final
# rename column
df.rename(columns={'year of construction of the building yyyy': 'year_of_construction'}, inplace=True)
# add _ to column names
df.columns = [col.replace(' ', '_') for col in df.columns]

# Define typical lifespans for components
component_lifespan = {
    'electrical cable': 30,
    'boiler': 20,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 40,
    'heat pump': 15,
}

# Define percentage distributions based on component
component_distributions = {
    "electrical cable": {"reuse": 0.10, "recycle": 0.40, "refurbish": 0.10, "disposal": 0.40},
    "boiler": {"reuse": 0.05, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.35},
    "water pipe": {"reuse": 0.20, "recycle": 0.30, "refurbish": 0.10, "disposal": 0.40},
    "air duct": {"reuse": 0.15, "recycle": 0.45, "refurbish": 0.15, "disposal": 0.25},
    "radiator": {"reuse": 0.05, "recycle": 0.25, "refurbish": 0.05, "disposal": 0.65},
    "heat pump": {"reuse": 0.05, "recycle": 0.55, "refurbish": 0.15, "disposal": 0.25},
}

# Define percentage distributions based on material
material_distributions = {
    "copper": {"reuse": 0.20, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.20},
    "pvc": {"reuse": 0.10, "recycle": 0.40, "refurbish": 0.10, "disposal": 0.40},
    "aluminum": {"reuse": 0.15, "recycle": 0.60, "refurbish": 0.10, "disposal": 0.15},
    "brass": {"reuse": 0.25, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.15},
    "zinc coating": {"reuse": 0.10, "recycle": 0.30, "refurbish": 0.10, "disposal": 0.50},
    "pex": {"reuse": 0.10, "recycle": 0.20, "refurbish": 0.10, "disposal": 0.60},
    "steel": {"reuse": 0.15, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.25},
}

# Current year
current_year = 2024

# Calculate age and determine post-use classification based on component and material
def classify_component(row):
    component = row['component']
    material = row['material']
    age = current_year - row['year_of_construction']
    lifespan = component_lifespan.get(component, 0)
    component_dist = component_distributions.get(component, {"reuse": 0, "recycle": 0, "refurbish": 0, "disposal": 0})
    material_dist = material_distributions.get(material, {"reuse": 0, "recycle": 0, "refurbish": 0, "disposal": 0})
    
    if age < lifespan:
        # Multiply component and material distributions
        reuse = component_dist['reuse'] * material_dist['reuse']
        recycle = component_dist['recycle'] * material_dist['recycle']
        refurbish = component_dist['refurbish'] * material_dist['refurbish']
        disposal = component_dist['disposal'] * material_dist['disposal']
        
        # Normalize to ensure the sum is 1
        total = reuse + recycle + refurbish + disposal
        if total > 0:
            row['reuse'] = reuse / total
            row['recycle'] = recycle / total
            row['refurbish'] = refurbish / total
            row['disposal'] = disposal / total
        else:
            row['reuse'] = 0
            row['recycle'] = 0
            row['refurbish'] = 0
            row['disposal'] = 1
    else:
        row['reuse'] = 0
        row['recycle'] = 0
        row['refurbish'] = 0
        row['disposal'] = 1
    
    return row

df = df.apply(classify_component, axis=1)

# Calculate end-of-life year
def calculate_end_of_life(row):
    lifespan = component_lifespan.get(row['component'], 0)
    return row['year_of_construction'] + lifespan

df['end_of_life_year'] = df.apply(calculate_end_of_life, axis=1)

# Define a function to create decade columns
def assign_decade(year):
    if year >= 2024 and year < 2034:
        return '2024-2034'
    elif year >= 2034 and year < 2044:
        return '2034-2044'
    elif year >= 2044 and year < 2054:
        return '2044-2054'
    elif year >= 2054 and year < 2064:
        return '2054-2064'
    elif year >= 2064 and year < 2074:
        return '2064-2074'
    else:
        return 'before 2024'

# Apply function to create the decade column
df['end_of_life_decade'] = df['end_of_life_year'].apply(assign_decade)

# Group by future_decade and other relevant columns and sum the material weights
df_aggregated = df.groupby(['building_class', 'building_system', 'component', 'material', 'end_of_life_decade']).agg({'material_weight': 'sum', 'reuse': 'mean', 'recycle': 'mean', 'refurbish': 'mean', 'disposal': 'mean'}).reset_index()

# Generate the reuse/recycle flows for each component and material
flows = []
for _, row in df_aggregated.iterrows():
    material_weight = row['material_weight']
    for category in ['reuse', 'recycle', 'refurbish', 'disposal']:
        flow = row.copy()
        flow['category'] = category
        flow['weight'] = material_weight * row[category]
        flows.append(flow)

df_flows = pd.DataFrame(flows)

# Aggregate flows by end-of-life decade and category
df_flows_aggregated = df_flows.groupby(['end_of_life_decade', 'category']).agg({'weight': 'sum'}).reset_index()

# Prepare the data for the Sankey diagram
# We need to create lists of unique labels for nodes and a way to map them to the source and target of links

# Create unique lists of labels
building_category = df['building_class'].unique().tolist()
building_system = df['building_system'].unique().tolist()
component = df['component'].unique().tolist()
material = df['material'].unique().tolist()
end_of_life_decade = df['end_of_life_decade'].unique().tolist()
categories = ['reuse', 'recycle', 'refurbish', 'disposal']

labels = building_category + building_system + component + material + end_of_life_decade + categories

# Create mappings from labels to indices
label_to_index = {label: index for index, label in enumerate(labels)}

# Create source and target lists for the Sankey diagram
source = []
target = []
value = []

# Mapping building category to building system
for _, row in df.iterrows():
    source.append(label_to_index[row['building_class']])
    target.append(label_to_index[row['building_system']])
    value.append(row['material_weight'])

# Mapping building system to component
for _, row in df.iterrows():
    source.append(label_to_index[row['building_system']])
    target.append(label_to_index[row['component']])
    value.append(row['material_weight'])

# Mapping component to material
for _, row in df.iterrows():
    source.append(label_to_index[row['component']])
    target.append(label_to_index[row['material']])
    value.append(row['material_weight'])

# Mapping material to end of life decade
for _, row in df.iterrows():
    source.append(label_to_index[row['material']])
    target.append(label_to_index[row['end_of_life_decade']])
    value.append(row['material_weight'])

# Mapping end of life decade to categories
for _, row in df_flows_aggregated.iterrows():
    source.append(label_to_index[row['end_of_life_decade']])
    target.append(label_to_index[row['category']])
    value.append(row['weight'])

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Building Material Flow Sankey Diagram", font_size=10)
fig.show()



In [ ]:
import pandas as pd
import plotly.graph_objects as go

# df = pd.DataFrame(data)
df = df_final
# rename column
df.rename(columns={'year of construction of the building yyyy': 'year_of_construction'}, inplace=True)
# add _ to column names
df.columns = [col.replace(' ', '_') for col in df.columns]

# Define typical lifespans for components
component_lifespan = {
    'electrical cable': 30,
    'boiler': 20,
    'water pipe': 50,
    'air duct': 25,
    'radiator': 40,
    'heat pump': 15,
}

# Define percentage distributions based on component
component_distributions = {
    "electrical cable": {"reuse": 0.10, "recycle": 0.40, "refurbish": 0.10, "disposal": 0.40},
    "boiler": {"reuse": 0.05, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.35},
    "water pipe": {"reuse": 0.20, "recycle": 0.30, "refurbish": 0.10, "disposal": 0.40},
    "air duct": {"reuse": 0.15, "recycle": 0.45, "refurbish": 0.15, "disposal": 0.25},
    "radiator": {"reuse": 0.05, "recycle": 0.25, "refurbish": 0.05, "disposal": 0.65},
    "heat pump": {"reuse": 0.05, "recycle": 0.55, "refurbish": 0.15, "disposal": 0.25},
}

# Define percentage distributions based on material
material_distributions = {
    "copper": {"reuse": 0.20, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.20},
    "pvc": {"reuse": 0.10, "recycle": 0.40, "refurbish": 0.10, "disposal": 0.40},
    "aluminum": {"reuse": 0.15, "recycle": 0.60, "refurbish": 0.10, "disposal": 0.15},
    "brass": {"reuse": 0.25, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.15},
    "zinc coating": {"reuse": 0.10, "recycle": 0.30, "refurbish": 0.10, "disposal": 0.50},
    "pex": {"reuse": 0.10, "recycle": 0.20, "refurbish": 0.10, "disposal": 0.60},
    "steel": {"reuse": 0.15, "recycle": 0.50, "refurbish": 0.10, "disposal": 0.25},
}

# Current year
current_year = 2024

# Calculate age and determine post-use classification based on component and material
def classify_component(row):
    component = row['component']
    material = row['material']
    age = current_year - row['year_of_construction']
    lifespan = component_lifespan.get(component, 0)
    component_dist = component_distributions.get(component, {"reuse": 0, "recycle": 0, "refurbish": 0, "disposal": 0})
    material_dist = material_distributions.get(material, {"reuse": 0, "recycle": 0, "refurbish": 0, "disposal": 0})
    
    if age < lifespan:
        # Multiply component and material distributions
        reuse = component_dist['reuse'] * material_dist['reuse']
        recycle = component_dist['recycle'] * material_dist['recycle']
        refurbish = component_dist['refurbish'] * material_dist['refurbish']
        disposal = component_dist['disposal'] * material_dist['disposal']
        
        # Normalize to ensure the sum is 1
        total = reuse + recycle + refurbish + disposal
        if total > 0:
            row['reuse'] = reuse / total
            row['recycle'] = recycle / total
            row['refurbish'] = refurbish / total
            row['disposal'] = disposal / total
        else:
            row['reuse'] = 0
            row['recycle'] = 0
            row['refurbish'] = 0
            row['disposal'] = 1
    else:
        row['reuse'] = 0
        row['recycle'] = 0
        row['refurbish'] = 0
        row['disposal'] = 1
    
    return row

df = df.apply(classify_component, axis=1)

# Generate the reuse/recycle flows for each component and material
flows = []
for _, row in df.iterrows():
    material_weight = row['material_weight']
    for category in ['reuse', 'recycle', 'refurbish', 'disposal']:
        flow = row.copy()
        flow['category'] = category
        flow['weight'] = material_weight * row[category]
        flows.append(flow)

df_flows = pd.DataFrame(flows)

# Aggregate flows by building class, building system, sub system, component, material, and category
df_flows_aggregated = df_flows.groupby(['building_class', 'building_system', 'sub_system', 'component', 'material', 'category']).agg({'weight': 'sum'}).reset_index()

# Prepare the data for the Sankey diagram
# We need to create lists of unique labels for nodes and a way to map them to the source and target of links

# Create unique lists of labels
building_category = df['building_class'].unique().tolist()
building_system = df['building_system'].unique().tolist()
sub_system = df['sub_system'].unique().tolist()
component = df['component'].unique().tolist()
material = df['material'].unique().tolist()
categories = ['reuse', 'recycle', 'refurbish', 'disposal']

labels = building_category + building_system + sub_system + component + material + categories

# Create mappings from labels to indices
label_to_index = {label: index for index, label in enumerate(labels)}

# Create source and target lists for the Sankey diagram
source = []
target = []
value = []

# Mapping building category to building system
for _, row in df.iterrows():
    source.append(label_to_index[row['building_class']])
    target.append(label_to_index[row['building_system']])
    value.append(row['material_weight'])

# Mapping building system to sub system
for _, row in df.iterrows():
    source.append(label_to_index[row['building_system']])
    target.append(label_to_index[row['sub_system']])
    value.append(row['material_weight'])

# Mapping sub system to component
for _, row in df.iterrows():
    source.append(label_to_index[row['sub_system']])
    target.append(label_to_index[row['component']])
    value.append(row['material_weight'])

# Mapping component to material
for _, row in df.iterrows():
    source.append(label_to_index[row['component']])
    target.append(label_to_index[row['material']])
    value.append(row['material_weight'])

# Mapping material to categories
for _, row in df_flows_aggregated.iterrows():
    source.append(label_to_index[row['material']])
    target.append(label_to_index[row['category']])
    value.append(row['weight'])

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Building Material Flow Sankey Diagram", font_size=10)
fig.show()
